In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
from tqdm import tqdm
import torch
import copy
from matplotlib.colors import hsv_to_rgb
import pandas as pd
import matplotlib as mpl
from matplotlib.widgets import LassoSelector
from matplotlib.path import Path
from sklearn.decomposition import PCA
from matplotlib.widgets import Slider
from scipy.ndimage import gaussian_filter1d
import sys
sys.path.append(os.path.abspath('..'))
from extract_experimental_psf import *
import bisect

In [ ]:
data = pd.read_csv("\\\\NAS_LOCCO\\Amaury\\DATA\\4_polar_MFM_these\\sperm_cricket_11.csv", delimiter=';')
frame = data['frame'].to_numpy().astype(int)
x = data['x'].to_numpy()
y = data['y'].to_numpy()
z = data['z'].to_numpy()
rho = data['rho'].to_numpy()
eta = data['eta'].to_numpy()
delta = data['delta'].to_numpy()
N_photons = data['N_photon'].to_numpy()
background_array_found = data['background_array_found'].to_numpy()
score = data['score'].to_numpy()
x_start = data['x_start'].to_numpy()
y_start = data['y_start'].to_numpy()
z_start = data['z_start'].to_numpy()
rho_start = data['rho_start'].to_numpy()
delta_start = data['delta_start'].to_numpy()

In [ ]:
%matplotlib inline
plt.rcParams['figure.figsize'] = [8,3]
hist = plt.hist(score, bins=1000)
plt.xlim((-4000000,-1000000))
plt.xlabel('Finale loss per PSF')
plt.ylabel('Occurences')

In [ ]:
plt.rcParams['figure.figsize'] = [8,3]
hist = plt.hist(N_photons, bins=80)
plt.xlim((0, 45000))
plt.xlabel('Photon number per PSF')
plt.ylabel('Occurences')

In [ ]:
%matplotlib inline
plt.rcParams['figure.figsize'] = [8,3]
hist = plt.hist(z, bins=200)
plt.xlim((-1000, 2000))
plt.xlabel('z')
plt.ylabel('Occurences')

In [ ]:
loss_thresh = -22000
mask1 = (z>200)&(z<700)
#mask1 = (score<loss_thresh) & (delta<130) & (delta>30) & (N_photons>500) & (N_photons<15000) & (z<1500) & (z>0)  
#zoom = (score<loss_thresh) & (delta<130) & (delta>30) & (N_photons>4000) & (N_photons<15000)  & (z<300) & (z>0)  

# Drift correction

In [ ]:
plt.scatter(x[mask1], y[mask1], c=frame[mask1], s=1)

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
vals = frame
sc = ax.scatter(x , y , c=vals , cmap='coolwarm', s=1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()
points = np.column_stack((x, y))
mask = np.zeros(len(x), dtype=bool)

def onselect(verts):
    global mask
    path = Path(verts)
    mask = path.contains_points(points) 
    print(mask)

lasso = LassoSelector(ax, onselect)
plt.show()

In [ ]:
def mean_x_per_20_frames(x, frame, window=20):

    bins = (frame // window).astype(int)
    n_bins = bins.max() + 1

    sum_x = np.bincount(bins, weights=x, minlength=n_bins)
    count = np.bincount(bins, minlength=n_bins)

    mean_x = np.zeros(n_bins)

    nonzero = count > 0
    mean_x[nonzero] = sum_x[nonzero] / count[nonzero]

    mean_x[~nonzero] = np.nan  # only real empty bins

    return mean_x

In [ ]:
mean_x = mean_x_per_20_frames(x[mask], frame[mask], 40)
mean_y = mean_x_per_20_frames(y[mask], frame[mask], 40)

In [ ]:
for i in range(len(mean_x)):
    if np.isnan(mean_x[i]):
        mean_x[i] = (mean_x[i-1]+mean_x[i+1])/2
        mean_y[i] = (mean_y[i-1]+mean_y[i+1])/2

In [ ]:
np.isnan(mean_x).any()
np.isnan(mean_y).any()

In [ ]:
%matplotlib inline
plt.rcParams['figure.figsize'] = [3,3]
xcorr = gaussian_filter1d(mean_x, 7)
xcorr = xcorr-xcorr[0]
plt.plot(xcorr)
plt.show()
ycorr = gaussian_filter1d(mean_y, 11)
ycorr = ycorr-ycorr[0]
plt.plot(ycorr)

In [ ]:
frame_bins = np.linspace(0, np.max(frame), len(xcorr)).astype(int)

In [ ]:
for ind, frame_ in tqdm(enumerate(frame)):
    idx = bisect.bisect_left(frame_bins, frame_)
    #print(frame_, idx, frame_bins[idx-1])
    x[ind]-=xcorr[idx-1]
    y[ind]-=ycorr[idx-1]
    #print(ind, frame_, x[ind], y[ind], xcorr[idx-1]*0.45, ycorr[idx-1]*0.6)

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
vals = frame
sc = ax.scatter(x , y , c=vals , cmap='coolwarm', s=0.1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [5, 5]
plt.rcParams.update({'font.size': 15})
hues = rho[mask1] / 180.0
hsv_colors = np.stack((hues, np.ones_like(hues), np.ones_like(hues)), axis=1)
rgb_colors = hsv_to_rgb(hsv_colors)
plt.scatter(x[mask1]/1000, y[mask1]/1000, c=rgb_colors, s=0.01)
plt.axis('equal')
#plt.xlim((10000, 22500))
#plt.ylim((2200, 14000))
plt.xlabel('x ($\\mu$m)')
plt.ylabel('y ($\\mu$m)')

# Half-circle colorbar rotated 90° clockwise
center_x, center_y = 20, -10
radius = 2
angles = np.linspace(0, np.pi, 200)  # rotated arc
for i in range(len(angles)-1):
    theta1, theta2 = angles[i], angles[i+1]
    hue = (np.degrees(theta1)) / 180.0  # Map to 0-1 range
    color = hsv_to_rgb([hue, 1, 1])
    arc_x = [center_x + radius * np.cos(theta1), center_x + radius * np.cos(theta2)]
    arc_y = [center_y + radius * np.sin(theta1), center_y + radius * np.sin(theta2)]
    plt.plot(arc_x, arc_y, color=color, lw=8, solid_capstyle='butt')

# Tick labels with offset
tick_angles = [180, 90, 0]  # Corresponding to hue range
label_offset = [-6.,1, -5]
plt.text(center_x, center_y, "$\\rho$ ($\\degree$)", ha='center', va='center', fontsize=15)
for i, ang in enumerate(tick_angles):
    rad = np.radians(ang)
    tx = center_x + (radius + label_offset[i]) * np.cos(np.pi-rad)
    ty = center_y + (radius + label_offset[i]) * np.sin(np.pi-rad)
    plt.text(tx, ty, f"{(ang)}", ha='center', va='center', fontsize=15)
#plt.xlim((6, 13.5))
#plt.ylim((12.8, 18.5))
#plt.savefig("imE.png", format="png")
plt.show()

In [ ]:
plt.scatter(eta, z, s=0.01)
plt.ylim((-300,1500))

In [ ]:
plt.scatter(rho, z, s=0.01)
plt.ylim((-300,1500))

In [ ]:
hh = plt.hist(delta, bins=100)
print(len(delta))

In [ ]:
hh = plt.hist(eta, bins=100)

In [ ]:
plt.scatter(rho, delta, s=0.01)

plt.show()

# Correlations

In [ ]:
%matplotlib inline
plt.rcParams['figure.figsize'] = [5, 5]
vals = rho[mask1]
plt.scatter(z[mask1], eta[mask1], s=0.1, c=vals, cmap='hsv')
plt.xlim((-500, 1500))

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [5, 5]
vals = rho
plt.scatter(z, delta, s=0.1, c=vals, cmap='hsv')
plt.xlim((-500, 1500))

In [ ]:
hh = plt.hist(rho[mask1], bins=100)

# Select ROI rho

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
norm = mpl.colors.Normalize(vmin=20., vmax=160.)
vals = rho[mask1]
sc = ax.scatter(x[mask1] , y[mask1], z[mask1], c=vals , cmap='hsv', norm=norm, s=0.1)
ax.axis('equal')
cb = plt.colorbar(sc)

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
norm = mpl.colors.Normalize(vmin=20., vmax=160.)
vals = rho[mask1]
sc = ax.scatter(x[mask1] , y[mask1], c=vals , cmap='hsv', norm=norm, s=0.1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()
points = np.column_stack((x, y))
mask2 = np.zeros(len(x), dtype=bool) 

def onselect(verts):
    global mask2
    path = Path(verts)
    mask2 = path.contains_points(points) & mask1
    print(mask2)

lasso = LassoSelector(ax, onselect)
plt.show()

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
norm = mpl.colors.Normalize(vmin=20., vmax=160.)
vals = eta[mask2]
sc = ax.scatter(x[mask2] , z[mask2], c=vals , cmap='coolwarm', norm=norm, s=0.1)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()
points = np.column_stack((x, z))
mask = np.zeros(len(x), dtype=bool) 

def onselect(verts):
    global mask
    path = Path(verts)
    mask = path.contains_points(points) & mask2
    print(mask)

lasso = LassoSelector(ax, onselect)
plt.show()

In [ ]:
angles = np.deg2rad(rho[mask])
bins = 18
counts, bin_edges = np.histogram(angles, bins=bins, density=True)

bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
width = bin_edges[1] - bin_edges[0]

# Duplicate with π shift
bin_centers_full = np.concatenate([bin_centers, bin_centers + np.pi])
counts_full = np.concatenate([counts, counts])

# ----- Figure layout -----
fig = plt.figure(figsize=(10, 5))

# Polar subplot
ax1 = fig.add_subplot(1, 2, 1, projection='polar')

ax1.bar(bin_centers_full, counts_full, width=width, alpha=0.8)

ax1.set_theta_zero_location("E")   # 0° to the right
ax1.set_theta_direction(1)         # anti-clockwise
ax1.set_yticklabels([])

# PCA
XY = np.column_stack((x[mask], y[mask]))
pca = PCA(n_components=1)
principal_coord = pca.fit_transform(XY).flatten()

# Sort along principal axis
idx = np.argsort(principal_coord)
x_sorted = principal_coord[idx]
z_sorted = z[mask].values[idx]   # use .values if it's pandas

# Moving average window size (adjust!)
window = 150

z_smooth = np.convolve(
    z_sorted,
    np.ones(window)/window,
    mode='valid'
)

# Correct matching x values
x_smooth = x_sorted[:len(z_smooth)]

# Cartesian subplot
ax2 = fig.add_subplot(1, 2, 2)
ax2.plot(x_smooth, z_smooth, color='red', linewidth=2)
ax2.scatter(principal_coord, z[mask], s=5, alpha=0.6)

ax2.set_xlabel("lateral coordinate")
ax2.set_ylabel("z")
ax2.set_aspect('equal')
ax2.set_title("lat–z scatter")
ax2.grid()
ax2.set_ylim(0,800)
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline
fig = plt.figure(figsize=(5,5))
ax = plt.subplot(111, projection='polar')

# convert to radians
theta = np.deg2rad(eta[mask])

# histogram in [0, π]
counts, bins = np.histogram(theta, bins=100, range=(0, np.pi))

# bin centers
theta_centers = (bins[:-1] + bins[1:]) / 2
width = np.diff(bins)

# duplicate at θ + π
theta_full = np.concatenate([theta_centers, theta_centers + np.pi])
counts_full = np.concatenate([counts, counts])
width_full = np.concatenate([width, width])

# plot
ax.bar(theta_full, counts_full, width=width_full)
# orientation
ax.set_theta_zero_location("N")   # 0° at the top
ax.set_theta_direction(-1)        # clockwise angles

plt.show()

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot(projection='3d')
norm = mpl.colors.Normalize(vmin=20., vmax=160.)
vals = eta[mask]
sc = ax.scatter(x[mask] , y[mask], z[mask], c=vals , cmap='coolwarm', norm=norm, s=0.01)
ax.axis('equal')
cb = plt.colorbar(sc)

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15,15]
fig = plt.figure()
ax = fig.add_subplot()
norm = mpl.colors.Normalize(vmin=0., vmax=180.)
vals = rho[mask]
sc = ax.scatter(x[mask] , y[mask], c=vals , cmap='hsv', norm=norm, s=0.01)
ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()

# Sliding window

In [ ]:
angle_to_analyse=eta

In [ ]:
%matplotlib qt
plt.rcParams['figure.figsize'] = [15, 15]

# Example: angle array in degrees (replace with yours)
theta = angle_to_analyse[mask]   # or use your own angle array

window_width = 10.0  # degrees
half_width = window_width / 2

fig, ax = plt.subplots()
plt.subplots_adjust(bottom=0.15)

norm = mpl.colors.Normalize(vmin=0., vmax=180.)
vals = angle_to_analyse

# Initial center
theta0 = 0.0

# Initial mask
def compute_mask(center):
    diff = (theta - center + 180) % 360 - 180
    return (np.abs(diff) <= half_width)&mask1

mask2 = compute_mask(theta0)

# Initial scatter
sc = ax.scatter(
    x[mask2],
    y[mask2],
    c=vals[mask2],
    cmap='coolwarm',
    norm=norm,
    s=1
)

ax.axis('equal')
cb = plt.colorbar(sc)
cb.ax.invert_yaxis()

# Slider axis
ax_slider = plt.axes([0.2, 0.05, 0.6, 0.03])
slider = Slider(
    ax=ax_slider,
    label='Angle center (deg)',
    valmin=half_width,
    valmax=180-half_width,
    valinit=theta0
)

# Update function
def update(val):
    center = slider.val
    mask2 = compute_mask(center)

    # Update scatter points
    sc.set_offsets(np.column_stack((x[mask2], y[mask2])))
    sc.set_array(vals[mask2])

    fig.canvas.draw_idle()

slider.on_changed(update)

plt.show()